## Data Processing

In [1]:
import pyarrow.parquet as pq
import pyarrow as pa
import pandas as pd

In [2]:
file_path = "..\\data\\Global Factor Data.parquet"
OUTPUT_FEATURES = "..\\data\\features_clean.parquet"
OUTPUT_TARGET = "..\\data\\target.parquet"

# Read full parquet file into Arrow Table
table = pq.read_table(file_path)
df = table.to_pandas()

display(df.head())

,obs_main,exch_main,common,primary_sec,gvkey,date,permno,permco,iid,id,...,dltnetis_mev,dstnetis_mev,dbnetis_mev,netis_mev,fincf_mev,ivol_capm_60m,beta_21d,beta_252d,rvol_252d,rvolhl_21d
0,1.0,1.0,1.0,1.0,040080,2024-12-31,22750.0,59043.0,02,22750.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.014801,0.003020,NaN
1,1.0,1.0,1.0,1.0,038617,2024-12-31,25327.0,59938.0,01,25327.0,...,0.000000,0.000000,0.000000,-0.000011,0.131983,NaN,2.325811,NaN,NaN,0.040160
2,1.0,1.0,1.0,1.0,144496,2024-12-31,15054.0,55100.0,02,15054.0,...,-0.058558,-0.619016,-0.677574,NaN,NaN,0.065632,1.058223,0.925554,0.015784,0.008192
3,1.0,1.0,1.0,1.0,037783,2024-12-31,20841.0,57760.0,02,20841.0,...,-0.376133,-0.368609,-0.744742,-0.092429,0.981088,0.337846,-0.025114,0.441205,0.093246,0.046725
4,1.0,1.0,1.0,1.0,038263,2024-12-31,25397.0,59969.0,01,25397.0,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.352928,NaN,NaN,0.005995


In [3]:
cols_to_remove = {
    # identifiers and keys
    "permno", "permco", "eom",

    # administrative and data quality flags
    "obs_main", "exch_main", "primary_sec", "common", "size_grp",
    "ret_lag_dif", "adjfct", "bidask", "comp_tpci", "crsp_shrcd",
    "comp_exchg", "crsp_exchcd", "source_crsp", "curcd",

    # target variable
    "ret_exc_lead1m",

    # current period returns, for being risk of data leakage
    "ret_exc", "ret", "ret_local",

    # raw unscaled accounting numbers
    "enterprise_value", "book_equity", "assets", "sales", "net_income",

    # exchange rates and industry codes
    "fx", "gics", "naics", "sic", "ff49",

    # raw price and volume data
    "prc", "prc_local", "prc_high", "prc_low", "shares", "tvol",
}

# read schema only without data loading
schema = pq.read_schema(file_path)
all_columns = schema.names

print(f"Total columns in file: {len(all_columns)}")

# determine columns
existing_remove = cols_to_remove & set(all_columns)

feature_cols = [
    col for col in all_columns
    if col not in cols_to_remove and col != "ret_exc_lead1m"
]

target_col = ["ret_exc_lead1m"] if "ret_exc_lead1m" in all_columns else []

print(f"Columns being removed: {len(existing_remove)}")
print(f"Feature columns kept: {len(feature_cols)}")
print(f"Target column present: {bool(target_col)}")

# stream parquet in batches
parquet_file = pq.ParquetFile(file_path)

feature_writer = None
target_writer = None

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):
    batch_table = pa.Table.from_batches([batch])

    # features
    feature_table = batch_table.select(feature_cols)

    if feature_writer is None:
        feature_writer = pq.ParquetWriter(
            OUTPUT_FEATURES,
            feature_table.schema,
            compression="snappy"
        )

    feature_writer.write_table(feature_table)

    # target
    if target_col:
        target_table = batch_table.select(target_col)

        if target_writer is None:
            target_writer = pq.ParquetWriter(
                OUTPUT_TARGET,
                target_table.schema,
                compression="snappy"
            )

        target_writer.write_table(target_table)

    print(f"Processed batch {i + 1} | rows: {batch_table.num_rows}")

# close writers
if feature_writer:
    feature_writer.close()

if target_writer:
    target_writer.close()

print(f"Features saved to: {OUTPUT_FEATURES}")
print(f"Target saved to: {OUTPUT_TARGET}")

# verify output
out_schema = pq.read_schema(OUTPUT_FEATURES)
print(f"Output feature columns: {len(out_schema.names)}")

Total columns in file: 443
Columns being removed: 37
Feature columns kept: 406
Target column present: True
Processed batch 1 | rows: 500000
Processed batch 2 | rows: 85595
Features saved to: ..\data\features_clean.parquet
Target saved to: ..\data\target.parquet
Output feature columns: 406


In [4]:
table_new = pq.read_table("..\\data\\features_clean.parquet")
df1 = table_new.to_pandas()
display(df1.head())

,gvkey,date,iid,id,excntry,me,me_company,dolvol,market_equity,div12m_me,...,dltnetis_mev,dstnetis_mev,dbnetis_mev,netis_mev,fincf_mev,ivol_capm_60m,beta_21d,beta_252d,rvol_252d,rvolhl_21d
0,040080,2024-12-31,02,22750.0,USA,50.46430,50.46430,8.829806e+08,50.46430,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.014801,0.003020,NaN
1,038617,2024-12-31,01,25327.0,USA,244.86336,244.86336,1.348946e+09,244.86336,NaN,...,0.000000,0.000000,0.000000,-0.000011,0.131983,NaN,2.325811,NaN,NaN,0.040160
2,144496,2024-12-31,02,15054.0,USA,97292.45328,97292.45328,8.219281e+10,97292.45328,0.028103,...,-0.058558,-0.619016,-0.677574,NaN,NaN,0.065632,1.058223,0.925554,0.015784,0.008192
3,037783,2024-12-31,02,20841.0,USA,11.59538,11.59538,1.185472e+09,11.59538,0.000000,...,-0.376133,-0.368609,-0.744742,-0.092429,0.981088,0.337846,-0.025114,0.441205,0.093246,0.046725
4,038263,2024-12-31,01,25397.0,USA,107.12000,107.12000,3.423243e+08,107.12000,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-0.352928,NaN,NaN,0.005995


In [5]:
table_new2 = pq.read_table("..\\data\\target.parquet")
df2 = table_new2.to_pandas()
display(df2.head())

,ret_exc_lead1m
0,NaN
1,-0.499329
2,0.164909
3,0.151342
4,-0.010028


In [6]:
INPUT_FILE = "..\\data\\features_clean.parquet"

parquet_file = pq.ParquetFile(INPUT_FILE)

results = []

for i, batch in enumerate(parquet_file.iter_batches(batch_size=500_000)):

    batch_table = pa.Table.from_batches([batch])
    df_batch = batch_table.to_pandas()

    nan_counts = df_batch.isnull().sum(axis=1)
    results.append(nan_counts)

    print(f"Batch {i + 1} processed | rows: {len(df_batch)}")

nan_per_row = pd.concat(results, ignore_index=True)

print(f"Total rows: {len(nan_per_row)}")
print(f"Rows with zero NaNs: {(nan_per_row == 0).sum()}")
print(f"Rows with at least one NaN: {(nan_per_row > 0).sum()}")
print(f"Average NaNs per row: {nan_per_row.mean():.2f}")
print(f"Max NaNs in a single row: {nan_per_row.max()}")

Batch 1 processed | rows: 500000
Batch 2 processed | rows: 85595
Total rows: 585595
Rows with zero NaNs: 0
Rows with at least one NaN: 585595
Average NaNs per row: 95.33
Max NaNs in a single row: 400


## Replacing NaNs